# OpenFold Mutation Batch

This notebook is for screening a set of point mutations against one base complex.

Use it when you want to answer: `which mutations look most promising for follow-up?`

Limits:
- this is a ranking workflow, not a direct affinity or ddG workflow
- use the output as a shortlist for deeper validation


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
HELPERS_DIR = PROJECT_ROOT / 'helpers'
if str(HELPERS_DIR) not in sys.path:
    sys.path.insert(0, str(HELPERS_DIR))

import pandas as pd
from of_notebook_lib import (
    RuntimeConfig,
    format_mutation_ranking,
    preview_molecules,
    run_mutation_scan,
    validate_molecules,
)

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)


## Runtime Setup

In [ ]:
runtime = RuntimeConfig(
    project_dir=Path('/home/jovyan/OpenFold'),
    openfold_prefix=Path('/home/jovyan/.mlspace/envs/openfold310'),
    results_dir=Path('/home/jovyan/OpenFold/results_refactored'),
    msa_cache_dir=Path('/home/jovyan/OpenFold/msa_cache/colabfold_msas'),
    triton_cache_dir=Path('/tmp/triton_cache'),
    fixed_msa_tmp_dir=Path('/tmp/of3_colabfold_msas'),
    use_fused_attention=False,
    use_deepspeed=False,
)

runtime


## User Input

Edit only this cell for normal usage.

In [ ]:
experiment_name = 'mutation_scan_test'

molecules = [
    {
        'molecule_type': 'protein',
        'chain_ids': ['A'],
        'sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG',
    },
    {
        'molecule_type': 'protein',
        'chain_ids': ['B'],
        'sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG',
    },
]

mutation_chain_id = 'B'
position_1based = 10
amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
include_wt = True

use_templates = True
use_msa_server = True
num_diffusion_samples = 1
num_model_seeds = 1
runner_yaml = None
inference_ckpt_path = None
inference_ckpt_name = None


## Input Preview

In [ ]:
issues = validate_molecules(molecules)
if issues:
    for issue in issues:
        print('INPUT ISSUE:', issue)
else:
    print('Input looks valid.')

display(preview_molecules(molecules))
print('mutation_chain_id =', mutation_chain_id)
print('position_1based =', position_1based)
print('num_mutations =', len(amino_acids))


## Run Mutation Batch

In [ ]:
result, mutation_summary, mutation_ranking = run_mutation_scan(
    runtime=runtime,
    experiment_name=experiment_name,
    molecules=molecules,
    mutation_chain_id=mutation_chain_id,
    position_1based=position_1based,
    amino_acids=amino_acids,
    include_wt=include_wt,
    use_templates=use_templates,
    use_msa_server=use_msa_server,
    num_diffusion_samples=num_diffusion_samples,
    num_model_seeds=num_model_seeds,
    runner_yaml=runner_yaml,
    inference_ckpt_path=inference_ckpt_path,
    inference_ckpt_name=inference_ckpt_name,
)

print('Return code:', result.return_code)
print('Run dir:', result.run_dir)
print('Summary dir:', result.summary_dir)


## Per-Sample Results

In [ ]:
display(
    result.samples_df[
        [
            'query_name',
            'mutation_label',
            'sample_name',
            'sample_ranking_score',
            'iptm',
            'ptm',
            'avg_plddt',
            'gpde',
            'has_clash',
        ]
    ].sort_values(['query_name', 'sample_ranking_score'], ascending=[True, False])
)


## Mutation Summary

In [ ]:
display(mutation_summary.sort_values(['is_wt', 'sample_ranking_score'], ascending=[False, False]))


## Ranked Mutations

In [ ]:
display(format_mutation_ranking(mutation_ranking))


## Key Files

In [ ]:
print('query.json        ->', result.query_path)
print('openfold log      ->', result.log_path)
print('output directory  ->', result.output_dir)
print('summary directory ->', result.summary_dir)
print('best report       ->', result.summary_dir / 'best_samples_report.txt')
